In [1]:
# predicting next-day direction (up/down classification)
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Load prepared dataset
df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/21_260311_features_for_training_df.csv")

In [3]:
features = ['pos','neut','neg','sentiment_score','rolling_sentiment',
            'sentiment_volatility','price_change','volatility','momentum',
            'rsi','macd','ema','sentiment_lag_1','price_lag_1','sentiment_lag_2',
            'price_lag_2','sentiment_lag_3','price_lag_3','sentiment_lag_4',
            'price_lag_4','sentiment_lag_5','price_lag_5']
target = "price_direction"   # 1 = up tomorrow, 0 = down

X = df[features].values
y = df[target].values

In [4]:
split_index = int(0.85 * len(df))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

scaler_X = MinMaxScaler()
X_train = scaler_X.fit_transform(X_train)   # fit on TRAIN only
X_test  = scaler_X.transform(X_test)         # transform test with same scaler
# no y scaling — y is 0/1 now

In [5]:
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test  = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(1, X_train.shape[2])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25, activation="relu"),
    Dense(1, activation="sigmoid")      # sigmoid for 0/1 probability
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=55, batch_size=16,
          validation_data=(X_test, y_test), verbose=1)

Epoch 1/55


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4840 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 2/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4740 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 3/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4740 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 4/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4740 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 5/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4740 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 6/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4740 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 7/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4740 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 8/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4740 - loss: nan - val_accuracy: 0.4691 - val_loss: nan
Epoch 9/55
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step -

In [6]:
from sklearn.metrics import accuracy_score
import numpy as np

# Baseline: always predict the majority class from the TRAIN set
majority = int(round(y_train.mean()))          # 1 if up-days more common, else 0
baseline_pred = np.full_like(y_test, majority)
base_acc = accuracy_score(y_test, baseline_pred)

model_acc = model.evaluate(X_test, y_test, verbose=0)[1]

print(f"Baseline (always predict {majority}) accuracy: {base_acc:.3f}")
print(f"Model accuracy: {model_acc:.3f}")
print(f"Up-day rate in test: {y_test.mean():.3f}")

Baseline (always predict 1) accuracy: 0.531
Model accuracy: 0.469
Up-day rate in test: 0.531
